# Demo: Semantic Kernel에 다양한 LLM 붙여보기

## 패키지 설치

In [ ]:
#r "nuget: Microsoft.Extensions.Configuration.Json, 9.*"
#r "nuget: Microsoft.SemanticKernel, 1.*"
#r "nuget: Microsoft.SemanticKernel.Connectors.Google, 1.*-*"
#r "nuget: Microsoft.SemanticKernel.Core, 1.*"


## 네임스페이스 지정

In [ ]:
using System.ClientModel;
using System.ComponentModel;
using System.IO;

using Microsoft.Extensions.Configuration;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using Microsoft.SemanticKernel.Connectors.Google;

using OpenAI;

using Kernel = Microsoft.SemanticKernel.Kernel;

## 액세스 키 로드

In [ ]:
var path = Path.Combine(Directory.GetCurrentDirectory(), "appsettings.Development.json");
var config = new ConfigurationBuilder().AddJsonFile(path).Build();

## Google Gemini 인스턴스 생성

In [ ]:
#pragma warning disable SKEXP0070
var kernel1 = Kernel.CreateBuilder()
                   .AddGoogleAIGeminiChatCompletion(
                        modelId: config["Google:Gemini:ModelName"],
                        apiKey: config["Google:Gemini:Token"])
                    .Build();
#pragma warning restore SKEXP0070

In [ ]:
var result = await kernel1.InvokePromptAsync("Who are you?");

Console.WriteLine(result);

## Azure OpenAI 인스턴스 생성

In [ ]:
var kernel2 = Kernel.CreateBuilder()
                    .AddAzureOpenAIChatCompletion(
                        endpoint: config["Azure:OpenAI:Endpoint"],
                        apiKey: config["Azure:OpenAI:Token"],
                        deploymentName: config["Azure:OpenAI:ModelName"])
                    .Build();

In [ ]:
var result = await kernel2.InvokePromptAsync("Who are you?");

Console.WriteLine(result);

## GitHub Models 인스턴스 생성

In [ ]:
var credentials = new ApiKeyCredential(config["GitHub:Phi:Token"]);
var options = new OpenAIClientOptions { Endpoint = new Uri(config["GitHub:Phi:Endpoint"]) };
var client = new OpenAIClient(credentials, options);

var kernel3 = Kernel.CreateBuilder()
                    .AddOpenAIChatCompletion(
                        modelId: config["GitHub:Phi:ModelName"],
                        openAIClient: client)
                    .Build();


In [ ]:
var result = await kernel3.InvokePromptAsync("Who are you?");

Console.WriteLine(result);

## 하나의 인스턴스에 모든 모델 붙여보기

In [ ]:
#pragma warning disable SKEXP0070

var kernel = Kernel.CreateBuilder()
                   .AddGoogleAIGeminiChatCompletion(
                        modelId: config["Google:Gemini:ModelName"],
                        apiKey: config["Google:Gemini:Token"],
                        serviceId: "gemini")
                    .AddAzureOpenAIChatCompletion(
                        endpoint: config["Azure:OpenAI:Endpoint"],
                        apiKey: config["Azure:OpenAI:Token"],
                        deploymentName: config["Azure:OpenAI:ModelName"],
                        serviceId: "openai")
                    .AddOpenAIChatCompletion(
                        modelId: config["GitHub:Phi:ModelName"],
                        openAIClient: client,
                        serviceId: "phi")
                    .Build();

#pragma warning restore SKEXP0070

In [ ]:
#pragma warning disable SKEXP0001

var serviceIds = new List<string> { "gemini", "openai", "phi" };
foreach (var serviceId in serviceIds)
{
    var settings = new PromptExecutionSettings
    {
        ServiceId = serviceId
    };
    var arguments = new KernelArguments(settings);

    var result = await kernel.InvokePromptAsync("Who are you?", arguments);

    Console.WriteLine(serviceId);
    Console.WriteLine("------");
    Console.WriteLine(result);
    Console.WriteLine("======");
    Console.WriteLine();
}

#pragma warning restore SKEXP0001